In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Pascal Siakam,Over,25.5,-137,2025-11-20,2025-11-19T22:33:48Z
1,PrizePicks,player_points,Pascal Siakam,Under,25.5,-137,2025-11-20,2025-11-19T22:33:48Z
2,PrizePicks,player_points,LaMelo Ball,Over,22.5,-137,2025-11-20,2025-11-19T22:33:48Z
3,PrizePicks,player_points,LaMelo Ball,Under,22.5,-137,2025-11-20,2025-11-19T22:33:48Z
4,PrizePicks,player_points,Miles Bridges,Over,22.5,-137,2025-11-20,2025-11-19T22:33:48Z


### Update projected starting lineups

In [23]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Top EVs for single bets

In [4]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 109 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
0,Josh Giddey,BetRivers,21.5,25.27,Over,120,0,5.06,0.421,High
1,Dereck Lively II,BetMGM,4.5,7.96,Over,-125,0,4.68,0.585,Low
2,Josh Giddey,BetRivers,20.5,25.27,Over,102,1,4.51,0.442,High
3,Pelle Larsson,BetRivers,10.5,13.30,Over,112,0,4.37,0.390,High
4,Landry Shamet,BetRivers,10.5,13.84,Over,100,0,4.20,0.420,High


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 87 players...
Error getting prediction for Coby White: float division by zero
Processing 78 players with valid predictions...
Generated 2843 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 116 combinations from 2843 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Dereck Lively II,Jerami Grant,4.5,23.5,7.96,17.27,over,under,0,7.27,0.364,Low,High
1,Dereck Lively II,Isaac Okoro,4.5,5.5,7.96,9.04,over,over,0,6.98,0.349,Low,High
2,Dereck Lively II,Josh Giddey,4.5,19.5,7.96,25.27,over,over,0,6.67,0.333,Low,High
3,Jerami Grant,Isaac Okoro,23.5,5.5,17.27,9.04,under,over,0,6.38,0.319,High,High
4,Jerami Grant,Josh Giddey,23.5,19.5,17.27,25.27,under,over,1,6.17,0.308,High,High
5,Landry Shamet,Isaac Okoro,9.5,5.5,13.84,9.04,over,over,0,5.88,0.294,High,High
6,Landry Shamet,Josh Giddey,9.5,19.5,13.84,25.27,over,over,0,5.57,0.279,High,High
7,Aaron Gordon,Landry Shamet,17.5,9.5,22.11,13.84,over,over,0,4.83,0.241,High,High
8,Aaron Gordon,Patrick Williams,17.5,5.5,22.11,7.53,over,over,0,4.01,0.200,High,Med
9,Aaron Gordon,Jeremiah Fears,17.5,14.5,22.11,18.11,over,over,0,3.89,0.195,High,High


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 123 players...
Error getting prediction for Coby White: float division by zero
Processing 115 players with valid predictions...
Generated 6231 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 171 combinations from 6231 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Dereck Lively II,Isaac Okoro,4.5,5.5,7.96,9.04,over,over,0,6.88,0.344,Low,High
1,Dereck Lively II,Josh Giddey,4.5,19.5,7.96,25.27,over,over,0,6.82,0.341,Low,High
2,Dereck Lively II,Mitchell Robinson,4.5,4.5,7.96,6.89,over,over,0,6.56,0.328,Low,Med
3,Tony Bradley,Isaac Okoro,4.5,5.5,7.06,9.04,over,over,0,5.93,0.297,Low,High
4,Landry Shamet,Isaac Okoro,9.5,5.5,13.84,9.04,over,over,0,5.80,0.290,High,High


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 87 players...
Error getting prediction for Coby White: float division by zero
Processing 78 players with valid predictions...
Generated 75094 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 51 combinations from 75094 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Dereck Lively II,Jerami Grant,Isaac Okoro,4.5,23.5,5.5,7.96,17.27,9.04,over,under,over,0,13.90,0.278,Low,High,High
1,Dereck Lively II,Jerami Grant,Josh Giddey,4.5,23.5,19.5,7.96,17.27,25.27,over,under,over,0,13.65,0.273,Low,High,High
2,Landry Shamet,Josh Giddey,Isaac Okoro,9.5,19.5,5.5,13.84,25.27,9.04,over,over,over,0,11.70,0.234,High,High,High
3,Aaron Gordon,Landry Shamet,Kevin Huerter,17.5,9.5,10.5,22.11,13.84,14.35,over,over,over,0,9.26,0.185,High,High,High
4,Aaron Gordon,Ayo Dosunmu,Kevin Huerter,17.5,11.5,10.5,22.11,14.95,14.35,over,over,over,0,8.04,0.161,High,High,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 123 players...
Error getting prediction for Coby White: float division by zero
Processing 115 players with valid predictions...
Generated 244242 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 76 combinations from 244242 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Dereck Lively II,Josh Giddey,Isaac Okoro,4.5,19.5,5.5,7.96,25.27,9.04,over,over,over,0,13.30,0.266,Low,High,High
1,Tony Bradley,Dereck Lively II,Isaac Okoro,4.5,4.5,5.5,7.06,7.96,9.04,over,over,over,0,13.01,0.260,Low,Low,High
2,Tony Bradley,Landry Shamet,Josh Giddey,4.5,9.5,19.5,7.06,13.84,25.27,over,over,over,0,11.21,0.224,Low,High,High
3,Landry Shamet,Mitchell Robinson,Jerami Grant,9.5,4.5,22.5,13.84,6.89,17.27,over,over,under,0,10.41,0.208,High,Med,High
4,Pelle Larsson,Mitchell Robinson,Jerami Grant,9.5,4.5,22.5,13.30,6.89,17.27,over,over,under,0,9.76,0.195,High,Med,High
